# Criteo Uplift Modeling: Individual Treatment Effect Estimation

**Single entry point.** Attach a Kaggle dataset containing `criteo-uplift-v2.1.csv`,
set `RUN_STAGE` below, then Run All. Every model, transform, and metric is
imported from `src/`; nothing is reimplemented in this notebook.

## The problem

An advertiser wants to know: *for which users does showing an ad actually
change their behaviour?* A response model answers a different, easier
question -- *who is likely to convert regardless* -- and targeting on it
wastes budget on users who would have converted anyway. **Uplift modeling**
instead estimates the **conditional average treatment effect (CATE)**,
`tau(x) = E[Y(1) - Y(0) | X = x]` -- `Y(1)`/`Y(0)` are the outcome a user
would have shown under treatment / control, only one of which is ever
observed -- so users can be ranked by the *incremental* effect of
treatment, not by their baseline propensity to convert.

## Objective

Compare four estimators of increasing sophistication on **CRITEO-UPLIFTv2.1**
(~13.9M rows, a real advertising A/B test) and report which ranks users by
incremental conversion most effectively, under an evaluation protocol that
never lets a model touch the data it is ultimately scored on:

| Model | Idea |
|---|---|
| **Response model** (LightGBM) | Non-causal comparator: predicts `P(Y\|X)`, ignores treatment entirely |
| **T-Learner** | Two independent outcome models, one per arm; `tau_hat = mu1_hat - mu0_hat` |
| **X-Learner** | Cross-fitted pseudo-outcome regression; corrects the T-Learner's imbalanced-arm bias |
| **Causal Forest** (`econml.grf.CausalForest`) | Honest random forest that splits directly on treatment-effect heterogeneity |

`Y` above is whichever outcome this run selects -- `conversion` by default,
or `visit` (see **Run parameters** below).

## Evaluation

- **Qini curve / Qini above random** -- primary ranking statistic
- **AUUC** (area under the uplift curve) -- a differently-weighted second view
- **Uplift@K** and a **decile breakdown** -- coarser, more interpretable slices
- **CATE distribution** -- is the model finding real heterogeneity, or a near-constant effect?

See `README.md`'s "Methodology notes" for the modeling decisions that matter
(D32 categorical handling, X-Learner fold-local preprocessing, Causal Forest
categorical encoding) and the four development notebooks for the full
derivation of each estimator:

| Notebook | Covers |
|---|---|
| `01_data_processing` | data contract, feature semantics, the split |
| `02_baseline_models` | response model, and why it is *not* a causal estimator |
| `03_uplift_models` | T-Learner, X-Learner, the fold-local leakage fix |
| `04_causal_forest` | Causal Forest, categorical encoding, final comparison |

## Why this notebook runs as stages, not one long script

`econml`'s `CausalForest` one-hot-encodes categorical features into a dense
matrix (~76 columns at the shipped `K=8`); at full CRITEO scale that matrix
alone is several GB, on top of the LightGBM-transformed matrices the other
three models use. Holding all of it in one kernel at once is a real OOM risk
on a standard Kaggle kernel. This notebook instead runs as **independent,
restart-safe stages** -- each stage loads only what it needs, saves its
expensive output as an artifact under `outputs/` (shared stages) or
`outputs/{outcome}/...` (model stages), and releases memory (`del` +
`gc.collect()`) before the next stage. Run All in
one session, or restart the kernel between stages (Data -> Baseline -> Uplift
-> Causal Forest -> Report) -- each stage picks up from the artifacts the
previous one saved, so nothing is retrained unnecessarily.

## Run parameters

- **`RUN_STAGE`** selects what to (re)compute this run. Every stage other than
  `"data"` first calls `ensure_data_artifacts()`, which loads existing data
  artifacts if they match the current `SAMPLE_ROWS`/`SEED`, or (re)computes
  them if missing or stale -- so `RUN_STAGE = "uplift"` alone works even on a
  fresh kernel, it just costs one data pass.

  | Value | Runs |
  |---|---|
  | `"data"` | data load, split, dataset/feature EDA |
  | `"baseline"` | response model |
  | `"uplift"` | T-Learner, X-Learner |
  | `"causal_forest"` | Causal Forest (the slow stage) |
  | `"report"` | final comparison + every visualization, **from artifacts only** -- fits nothing, so it is the stage to run after a kernel restart to see results without retraining |
  | `"all"` | every stage above, in order, in one session |
- **`SAMPLE_ROWS = None`** runs the complete ~13.9M-row experiment. Set e.g.
  `SAMPLE_ROWS = 1_000_000` for a fast end-to-end check first -- the sample is
  stratified on `(treatment, conversion)`, and every model sees the same rows.
- **`OUTCOME = "conversion"`** (default) or `"visit"` selects which column every
  model stage reads as `Y` (see `src.data.resolve_outcome`). `conversion` is the
  primary, business-oriented experiment; `visit` is a secondary sensitivity
  analysis on the exact same row partition -- see the outcome comparison in
  Stage 1 for why. Model, evaluation, and baseline/uplift/causal_forest/report
  artifacts are kept in separate `outputs/{outcome}/...` subtrees so the two
  never collide; `outputs/data/` and `outputs/preprocessing/` are shared, since
  neither depends on which outcome is selected.
- **`RUN_CAUSAL_FOREST = True`**: the Causal Forest is the bottleneck
  (`econml`'s `CausalForest` runs with `n_jobs=1`, required for reproducible
  predictions, so a full-data fit can take hours). Set `False` to skip it --
  the report stage still shows every other model and marks Causal Forest as
  pending rather than fabricate a result.

In [ ]:
RUN_STAGE = "all"          # "data" | "baseline" | "uplift" | "causal_forest" | "report" | "all"
SAMPLE_ROWS = None          # None = full dataset; e.g. 1_000_000 for a fast run
SEED = 42
OUTCOME = "conversion"      # "conversion" (primary) | "visit" (secondary sensitivity analysis --
                             # see docs/secondary_visit_outcome_experiment_plan.md). Resolved and
                             # validated just below, once resolve_outcome is imported.
RUN_CAUSAL_FOREST = True    # set False to skip the slowest stage

_VALID_STAGES = ("data", "baseline", "uplift", "causal_forest", "report", "all")
assert RUN_STAGE in _VALID_STAGES, f"RUN_STAGE={RUN_STAGE!r} must be one of {_VALID_STAGES}"

## Stage 0 -- Environment validation

Always runs, regardless of `RUN_STAGE`: locate the repository (this notebook
does not assume the kernel's working directory), make `src/` importable, and
verify every dependency actually imports before any expensive work starts.

In [ ]:
import sys, platform
from pathlib import Path


def find_repo_root() -> Path:
    bases = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working"), Path("/kaggle/input")]
    for base in bases:
        if not base.is_dir():
            continue
        if (base / "src" / "data.py").is_file():
            return base
        for child in sorted(p for p in base.iterdir() if p.is_dir()):
            if (child / "src" / "data.py").is_file():
                return child
    return None


REPO_ROOT = find_repo_root()

if REPO_ROOT is None and Path("/kaggle/working").is_dir():
    # Repo not attached -- try cloning it (requires Internet enabled in kernel settings).
    import subprocess
    target = Path("/kaggle/working/causal-uplift-modeling")
    print("Repository not found; attempting clone into", target)
    result = subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/arthur105204/causal-uplift-modeling.git", str(target)],
        capture_output=True, text=True,
    )
    print(result.stdout or result.stderr)
    REPO_ROOT = find_repo_root()

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate the repository. Either attach it as a Kaggle dataset, "
        "clone it into /kaggle/working, or enable Internet in the kernel settings "
        "so this cell can clone it automatically."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("python     :", platform.python_version())
print("repo root  :", REPO_ROOT)

In [ ]:
import importlib

REQUIRED = ["numpy", "pandas", "sklearn", "lightgbm", "econml", "pyarrow", "matplotlib", "yaml", "joblib"]
missing = []
for name in REQUIRED:
    try:
        module = importlib.import_module(name)
        print(f"{name:12s} {getattr(module, '__version__', 'n/a')}")
    except ImportError as exc:
        missing.append(name)
        print(f"{name:12s} MISSING ({exc})")

if missing:
    print("\nInstalling missing packages...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
    print("Re-run this cell to confirm. If econml is missing, install with: pip install econml==0.17.0")
else:
    print("\nAll dependencies present.")

In [ ]:
import gc
import time

# Everything below comes from src/ -- no model code is defined in this notebook.
from src.data import (
    CATEGORICAL_FEATURES, CONTINUOUS_FEATURES, FEATURE_COLUMNS,
    PRIMARY_OUTCOME, SECONDARY_OUTCOME, TREATMENT_COLUMN,
    basic_summary, load_config, load_csv, load_parquet, on_kaggle,
    resolve_csv_path, resolve_outcome, save_parquet,
)
from src.preprocessing import (
    CausalForestCategoricalEncoder, LightGBMFeatureTransform, train_validation_test_split,
)
from src.models import (
    fit_causal_forest, fit_response_model, fit_t_learner, fit_x_learner,
    predict, predict_causal_forest_tau,
)
from src.evaluation import compute_ate, evaluate_ranking, response_diagnostics
from src.artifacts import (
    artifact_is_fresh, artifact_root, config_fingerprint, experiment_metadata,
    load_csv_artifact, load_json, load_pickle, save_csv, save_json, save_pickle, stage_dir,
)

CONFIG = load_config()
ARTIFACT_ROOT = artifact_root()
OUTCOME_COLUMN = resolve_outcome(OUTCOME)  # fails fast here if OUTCOME is misconfigured
print("on kaggle     :", on_kaggle())
print("artifact root :", ARTIFACT_ROOT)
print("outcome       :", OUTCOME_COLUMN)
print("imports OK")

In [ ]:
# Reproducibility fingerprint -- what produced this run, in case a reviewer
# needs to match a result back to an exact config/commit.
import platform as _platform
import subprocess as _subprocess


def _git_commit() -> str:
    try:
        result = _subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT,
            capture_output=True, text=True, timeout=5,
        )
        return result.stdout.strip() or "unknown"
    except Exception:
        return "unknown (not a git checkout, or git unavailable)"


ENV_INFO = {
    "python_version": _platform.python_version(),
    "platform": _platform.platform(),
    "git_commit": _git_commit(),
    "seed": SEED,
    "sample_rows": SAMPLE_ROWS,
    "run_stage": RUN_STAGE,
    "outcome_column": OUTCOME_COLUMN,
}
for key, value in ENV_INFO.items():
    print(f"{key:15s}: {value}")

In [ ]:
# Shared helper: every model-producing stage (baseline / uplift / causal
# forest) packages its test-set evaluation identically -- one prediction
# artifact, one metrics artifact, one set of curve artifacts -- so the report
# stage can reconstruct the full comparison from disk without importing any
# stage-specific code. This is notebook I/O plumbing around
# src.evaluation.evaluate_ranking, not a reimplementation of it.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def package_ranking_artifacts(
    model_dir, label, val_metrics, test_metrics,
    val_scores, test_scores, T_val, Y_val, T_test, Y_test,
    row_id_val, row_id_test, runtime_seconds, extra_metrics=None,
):
    predictions = pd.concat([
        pd.DataFrame({
            "row_id": np.asarray(row_id_val), "partition": "validation",
            "score": np.asarray(val_scores, dtype=np.float64),
            "treatment": np.asarray(T_val, dtype=np.float64), "outcome": np.asarray(Y_val, dtype=np.float64),
        }),
        pd.DataFrame({
            "row_id": np.asarray(row_id_test), "partition": "test",
            "score": np.asarray(test_scores, dtype=np.float64),
            "treatment": np.asarray(T_test, dtype=np.float64), "outcome": np.asarray(Y_test, dtype=np.float64),
        }),
    ], ignore_index=True)
    save_parquet(predictions, model_dir / "predictions.parquet")
    save_csv(test_metrics.qini_curve, model_dir / "qini_curve.csv")
    save_csv(test_metrics.uplift_curve, model_dir / "uplift_curve.csv")
    save_csv(test_metrics.decile_table, model_dir / "decile_table.csv")
    metrics = {
        "label": label,
        "runtime_seconds": runtime_seconds,
        "n_test": test_metrics.n,
        "val_qini_above_random": val_metrics.qini_above_random,
        "val_auuc_above_random": val_metrics.auuc_above_random,
        "test_qini_above_random": test_metrics.qini_above_random,
        "test_auuc_above_random": test_metrics.auuc_above_random,
        "test_qini_area": test_metrics.qini_area,
        "test_auuc_area": test_metrics.auuc_area,
        "test_uplift_at_k": test_metrics.uplift_at_k,
    }
    if extra_metrics:
        metrics.update(extra_metrics)
    save_json(metrics, model_dir / "metrics.json")
    return metrics


def load_ranking_artifact(model_dir):
    return {
        "metrics": load_json(model_dir / "metrics.json"),
        "qini_curve": load_csv_artifact(model_dir / "qini_curve.csv"),
        "uplift_curve": load_csv_artifact(model_dir / "uplift_curve.csv"),
        "decile_table": load_csv_artifact(model_dir / "decile_table.csv"),
        "predictions": load_parquet(model_dir / "predictions.parquet"),
    }

## Stage 1 -- Data processing

Loads the raw CSV (auto-discovering the attached Kaggle dataset slug),
verifies the causal contract (`X = f0..f11`, `T = treatment`, `Y = conversion` by
default or `visit` when `OUTCOME = "visit"`),
computes the population ATE, optionally subsamples, and produces the seeded
70/15/15 train/validation/test split every later stage reads.

**Feature semantics (D32).** All twelve `f0`-`f11` columns are stored as
`float64`, but physical storage does not imply semantic type: `f0`, `f2`,
`f7`, `f10` are genuinely **continuous**; `f1`, `f3`, `f4`, `f5`, `f6`, `f8`,
`f9`, `f11` are **categorical numeric tokens** with no ordinal meaning.
`exposure` is post-assignment and is loaded only for the contract check --
never as a feature.

**Artifacts written** under `outputs/data/`: `dataset_summary.json`,
`feature_summary.csv`, `split_summary.json`, `run_config.json`, and
`train.parquet` / `validation.parquet` / `test.parquet` (the raw, unencoded
splits every other stage reads instead of re-parsing the CSV). Every other
stage calls `ensure_data_artifacts()` first and reuses these unchanged if
they match the current `SAMPLE_ROWS`/`SEED` -- this split is computed once,
stratified on `(treatment, conversion)`, and reused **unchanged** regardless of
which `OUTCOME` a run selects: both `conversion` and `visit` are already
present in every saved partition (see the outcome comparison below), so
switching outcomes only changes which column is read as `Y`, never the rows.

In [ ]:
def ensure_data_artifacts():
    # Load-or-compute the data-stage artifacts. Returns the data signature
    # every downstream stage's own artifacts are checked against.
    data_dir = stage_dir("data")
    split_cfg = CONFIG["split"]
    signature = config_fingerprint(SAMPLE_ROWS, SEED, split_cfg, CONFIG["data"])
    run_config_path = data_dir / "run_config.json"
    parquet_paths = [data_dir / f"{name}.parquet" for name in ("train", "validation", "test")]

    if artifact_is_fresh(run_config_path, signature) and all(p.is_file() for p in parquet_paths):
        print(f"Using cached data artifacts ({data_dir}), signature={signature}")
        return signature

    print("Computing data artifacts (missing or stale) ...")
    stage_start = time.perf_counter()

    csv_path = resolve_csv_path()
    frame = load_csv(csv_path)

    missing_features = [f for f in FEATURE_COLUMNS if f not in frame.columns]
    assert not missing_features, f"missing features: {missing_features}"
    assert TREATMENT_COLUMN in frame.columns, f"missing treatment column {TREATMENT_COLUMN!r}"
    assert PRIMARY_OUTCOME in frame.columns, f"missing outcome column {PRIMARY_OUTCOME!r}"
    assert set(frame[TREATMENT_COLUMN].unique()) <= {0, 1}, "treatment must be binary"
    assert set(frame[PRIMARY_OUTCOME].unique()) <= {0, 1}, "conversion must be binary"

    summary = basic_summary(frame)
    ate = compute_ate(frame[TREATMENT_COLUMN], frame[PRIMARY_OUTCOME])
    by_treatment = frame.groupby(TREATMENT_COLUMN)[[PRIMARY_OUTCOME, SECONDARY_OUTCOME]].mean()

    if SAMPLE_ROWS is not None and SAMPLE_ROWS < len(frame):
        from sklearn.model_selection import train_test_split as _tts
        strata = frame[TREATMENT_COLUMN].astype(str) + "_" + frame[PRIMARY_OUTCOME].astype(str)
        frame, _ = _tts(frame, train_size=SAMPLE_ROWS, random_state=SEED, stratify=strata)
        frame = frame.reset_index(drop=True)

    train_frame, val_frame, test_frame = train_validation_test_split(
        frame,
        train_fraction=split_cfg["train_fraction"],
        validation_fraction=split_cfg["validation_fraction"],
        test_fraction=split_cfg["test_fraction"],
        seed=SEED,
    )
    keep_cols = list(FEATURE_COLUMNS) + [TREATMENT_COLUMN, PRIMARY_OUTCOME, SECONDARY_OUTCOME]
    partitions = {"train": train_frame, "validation": val_frame, "test": test_frame}
    split_summary = {}
    for name, part in partitions.items():
        out = part.loc[:, keep_cols].reset_index(drop=True)
        out.insert(0, "row_id", out.index.to_numpy())
        save_parquet(out, data_dir / f"{name}.parquet")
        split_summary[name] = {
            "n_rows": int(len(part)),
            "treatment_rate": float(part[TREATMENT_COLUMN].mean()),
            "conversion_rate": float(part[PRIMARY_OUTCOME].mean()),
            "visit_rate": float(part[SECONDARY_OUTCOME].mean()),
        }

    feature_rows = []
    for f in CONTINUOUS_FEATURES:
        s = train_frame[f].astype("float64")
        desc = s.describe(percentiles=[0.25, 0.5, 0.75])
        feature_rows.append({
            "feature": f, "kind": "continuous", "n_unique": int(s.nunique()),
            "mean": float(desc["mean"]), "std": float(desc["std"]), "min": float(desc["min"]),
            "p25": float(desc["25%"]), "p50": float(desc["50%"]), "p75": float(desc["75%"]), "max": float(desc["max"]),
            "top_category": None, "top_category_share": None,
        })
    for f in CATEGORICAL_FEATURES:
        vc = train_frame[f].astype("float64").value_counts()
        feature_rows.append({
            "feature": f, "kind": "categorical", "n_unique": int(vc.shape[0]),
            "mean": None, "std": None, "min": None, "p25": None, "p50": None, "p75": None, "max": None,
            "top_category": float(vc.index[0]), "top_category_share": float(vc.iloc[0] / len(train_frame)),
        })
    save_csv(pd.DataFrame(feature_rows), data_dir / "feature_summary.csv")

    dataset_summary = {
        "n_rows": summary["n_rows"], "n_cols": summary["n_cols"],
        "n_features": len(FEATURE_COLUMNS),
        "continuous_features": list(CONTINUOUS_FEATURES), "categorical_features": list(CATEGORICAL_FEATURES),
        "treatment_counts": {str(k): int(v) for k, v in summary["treatment_counts"].items()},
        "conversion_rate": summary["conversion_rate"],
        "visit_rate": float(frame[SECONDARY_OUTCOME].mean()),
        "conversion_rate_by_treatment": {str(k): float(v) for k, v in by_treatment[PRIMARY_OUTCOME].items()},
        "visit_rate_by_treatment": {str(k): float(v) for k, v in by_treatment[SECONDARY_OUTCOME].items()},
        "ate": {"ate": ate.ate, "se": ate.se, "ci_95_low": ate.ci_95_low, "ci_95_high": ate.ci_95_high,
                "relative_lift": ate.relative_lift},
        "csv_path": str(csv_path), "csv_size_mb": csv_path.stat().st_size / 1024**2,
        "sample_rows_used": None if SAMPLE_ROWS is None else int(len(frame)),
    }
    save_json(dataset_summary, data_dir / "dataset_summary.json")
    save_json(split_summary, data_dir / "split_summary.json")
    save_json({
        "data_signature": signature, "seed": SEED, "sample_rows": SAMPLE_ROWS,
        "split": split_cfg, "env": ENV_INFO,
        "runtime_seconds": time.perf_counter() - stage_start,
    }, run_config_path)

    del frame, train_frame, val_frame, test_frame, by_treatment
    gc.collect()
    print(f"Data artifacts written to {data_dir} in {time.perf_counter() - stage_start:.1f}s")
    return signature


DATA_SIGNATURE = ensure_data_artifacts()

In [ ]:
if RUN_STAGE in ("data", "all"):
    data_dir = stage_dir("data")
    dataset_summary = load_json(data_dir / "dataset_summary.json")
    split_summary = load_json(data_dir / "split_summary.json")

    summary_table = pd.DataFrame({
        "rows": {k: v["n_rows"] for k, v in split_summary.items()},
        "treatment_rate": {k: v["treatment_rate"] for k, v in split_summary.items()},
        "conversion_rate": {k: v["conversion_rate"] for k, v in split_summary.items()},
        "visit_rate": {k: v["visit_rate"] for k, v in split_summary.items()},
    })
    print(f"Rows (full)  : {dataset_summary['n_rows']:,}")
    print(f"Features     : {dataset_summary['n_features']} "
          f"({len(dataset_summary['continuous_features'])} continuous, "
          f"{len(dataset_summary['categorical_features'])} categorical)")
    print(f"Treatment    : {dataset_summary['treatment_counts']}")
    print(f"Conversion   : {dataset_summary['conversion_rate']:.5f} overall  "
          f"(treated {dataset_summary['conversion_rate_by_treatment'].get('1', float('nan')):.5f} / "
          f"control {dataset_summary['conversion_rate_by_treatment'].get('0', float('nan')):.5f})")
    print(f"Visit        : {dataset_summary['visit_rate']:.5f} overall")
    ate_info = dataset_summary["ate"]
    print(f"Population ATE : {ate_info['ate']:.6f}  (95% CI {ate_info['ci_95_low']:.6f} .. {ate_info['ci_95_high']:.6f})")
    display(summary_table.round(5))
else:
    print(f"Data EDA skipped (RUN_STAGE={RUN_STAGE!r}); artifacts still ensured above.")

In [ ]:
if RUN_STAGE in ("data", "all"):
    treated_n = dataset_summary["treatment_counts"].get("1", 0)
    control_n = dataset_summary["treatment_counts"].get("0", 0)
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    ax.bar(["Control", "Treatment"], [control_n, treated_n], color=["#888888", "#2b6cb0"])
    ax.set_ylabel("Rows")
    ax.set_title("Treatment vs. control")
    for i, v in enumerate([control_n, treated_n]):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom")
    fig.tight_layout()
    plt.show()

In [ ]:
if RUN_STAGE in ("data", "all"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, key, title in [
        (ax1, "conversion_rate_by_treatment", "Conversion rate by arm"),
        (ax2, "visit_rate_by_treatment", "Visit rate by arm"),
    ]:
        rates = dataset_summary[key]
        ax.bar(["Control", "Treatment"], [rates.get("0", 0.0), rates.get("1", 0.0)], color=["#888888", "#2b6cb0"])
        ax.set_title(title)
        ax.set_ylabel("Rate")
    fig.tight_layout()
    plt.show()

### Outcome comparison: conversion vs. visit

**Observation.** `conversion` -- the final, business-relevant action -- is
the outcome this project optimizes for, but it is a rare event. `visit` is
a much more common intermediate action on the same causal path (every
conversion is preceded by a visit; most visits do not convert). Same rows,
same treatment assignment, same causal question -- just a less sparse label
-- which is what makes it a natural secondary outcome for a sensitivity
analysis rather than an unrelated second experiment.

**Quantitative evidence.**

In [ ]:
if RUN_STAGE in ("data", "all"):
    def _outcome_comparison_row(label, overall_rate, rate_by_treatment):
        treatment_rate = rate_by_treatment.get("1", float("nan"))
        control_rate = rate_by_treatment.get("0", float("nan"))
        return {
            "outcome": label,
            "overall_rate": overall_rate,
            "treatment_rate": treatment_rate,
            "control_rate": control_rate,
            "absolute_difference": treatment_rate - control_rate,
        }

    outcome_comparison = pd.DataFrame([
        _outcome_comparison_row(
            "conversion", dataset_summary["conversion_rate"], dataset_summary["conversion_rate_by_treatment"]
        ),
        _outcome_comparison_row(
            "visit", dataset_summary["visit_rate"], dataset_summary["visit_rate_by_treatment"]
        ),
    ]).set_index("outcome")

    prevalence_ratio = dataset_summary["visit_rate"] / dataset_summary["conversion_rate"]
    print(f"visit is {prevalence_ratio:.1f}x more prevalent than conversion in this dataset")
    print(f"this run's selected outcome (OUTCOME_COLUMN): {OUTCOME_COLUMN!r}")
    display(outcome_comparison.round(6))

**Modeling implication.** An outcome this sparse (`conversion`) means every
stage that estimates a treatment effect by differencing two outcome models
(T-Learner) or regressing an imputed pseudo-outcome (X-Learner) is working
with very few positive labels per arm -- estimation variance is driven by
how rare the event is, not by row count alone. `visit`'s much higher
prevalence gives the same estimators, unchanged, a denser signal to learn
from, which is precisely why comparing model rankings under both outcomes is
an informative check on whether a conclusion reached under `conversion` is a
real effect or an artifact of outcome sparsity.

**Interpretation.** `visit` does not replace `conversion` as the primary,
business-oriented experiment -- it runs in parallel, on the exact same
physical train/validation/test row partition (the split above is fit once,
stratified on `(treatment, conversion)`, and reused unchanged for both;
only which column is read as `Y` changes). Its results are reported
alongside `conversion`'s as a secondary sensitivity analysis, never as a
substitute conclusion.

### Feature analysis

Continuous features (`f0`, `f2`, `f7`, `f10`) and categorical numeric tokens
(`f1`, `f3`, `f4`, `f5`, `f6`, `f8`, `f9`, `f11`, per D32) get different
diagnostics. Cardinality and summary statistics below are computed on the
**full train partition** (cheap, done once inside `ensure_data_artifacts`);
the distribution plots use a bounded random sample of it so this section
never materializes the full dataset into a second in-memory copy.

In [ ]:
if RUN_STAGE in ("data", "all"):
    feature_summary = load_csv_artifact(data_dir / "feature_summary.csv")

    train_full = load_parquet(data_dir / "train.parquet")
    PLOT_SAMPLE_ROWS = min(len(train_full), 500_000)
    plot_sample = train_full.sample(n=PLOT_SAMPLE_ROWS, random_state=SEED) if PLOT_SAMPLE_ROWS < len(train_full) else train_full
    del train_full
    gc.collect()
    print(f"Plotting sample: {len(plot_sample):,} rows drawn from the train partition")

    cont_summary = feature_summary[feature_summary["kind"] == "continuous"].set_index("feature")
    display(cont_summary[["n_unique", "mean", "std", "min", "p25", "p50", "p75", "max"]].round(4))

    fig, axes = plt.subplots(1, len(CONTINUOUS_FEATURES), figsize=(4 * len(CONTINUOUS_FEATURES), 3))
    for ax, f in zip(axes, CONTINUOUS_FEATURES):
        ax.hist(plot_sample[f].astype("float64"), bins=50, color="#2b6cb0")
        ax.set_title(f)
    fig.suptitle("Continuous feature distributions (train sample)")
    fig.tight_layout()
    plt.show()

In [ ]:
if RUN_STAGE in ("data", "all"):
    cat_summary = feature_summary[feature_summary["kind"] == "categorical"].set_index("feature")
    display(cat_summary[["n_unique", "top_category", "top_category_share"]].round(4))

    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, f in zip(axes.ravel(), CATEGORICAL_FEATURES):
        top = plot_sample[f].astype("float64").value_counts().head(10)
        ax.bar(range(len(top)), top.to_numpy(), color="#2b6cb0")
        ax.set_xticks([])
        ax.set_title(f"{f} (top 10 of {int(cat_summary.loc[f, 'n_unique'])})", fontsize=9)
    fig.suptitle("Categorical feature top-category frequency (train sample)")
    fig.tight_layout()
    plt.show()

    del plot_sample
    gc.collect()

## Stage 2 -- Baseline (response model)

A plain LightGBM binary classifier on the selected outcome (`OUTCOME_COLUMN`
-- `conversion` by default, or `visit`), ignoring treatment entirely. It is
the **non-causal comparator**: strong response-ranking performance here says
nothing about *causal* ranking quality, which is the entire point of running
it alongside the uplift models.

**Artifacts written** under `outputs/{OUTCOME_COLUMN}/baseline/` (e.g.
`outputs/conversion/baseline/`): `model.pkl`, `metrics.json`,
`predictions.parquet` (validation + test scores), `qini_curve.csv`,
`uplift_curve.csv`, `decile_table.csv`, and `roc_curve.csv` / `pr_curve.csv`
/ `calibration_curve.csv` (validation-set diagnostics -- meaningful here
because this model's output is a calibrated probability; the uplift models'
outputs are not).

In [ ]:
def ensure_lgbm_transform():
    # Load-or-fit the LightGBM categorical-feature transform. Used by the
    # response model and the T-Learner; the X-Learner fits its own fold-local
    # transforms internally and does not use this one (see Stage 3).
    prep_dir = stage_dir("preprocessing")
    meta_path = prep_dir / "lgbm_transform_metadata.json"
    transform_path = prep_dir / "lgbm_transform.joblib"
    if artifact_is_fresh(meta_path, DATA_SIGNATURE) and transform_path.is_file():
        return load_pickle(transform_path)

    train_frame = load_parquet(stage_dir("data") / "train.parquet")
    transform = LightGBMFeatureTransform().fit(train_frame)
    save_pickle(transform, transform_path)
    save_json({
        "data_signature": DATA_SIGNATURE,
        "continuous_features": list(CONTINUOUS_FEATURES),
        "categorical_features": list(CATEGORICAL_FEATURES),
    }, meta_path)
    del train_frame
    gc.collect()
    return transform

In [ ]:
def ensure_baseline_artifacts():
    baseline_dir = stage_dir("baseline", outcome=OUTCOME_COLUMN)
    meta_path = baseline_dir / "metrics.json"
    if (artifact_is_fresh(meta_path, DATA_SIGNATURE, outcome=OUTCOME_COLUMN)
            and (baseline_dir / "model.pkl").is_file()):
        print(f"Using cached baseline artifacts ({baseline_dir})")
        return load_json(meta_path)

    print("Fitting response model (cache missing or stale) ...")
    data_dir = stage_dir("data")
    transform = ensure_lgbm_transform()
    train_frame = load_parquet(data_dir / "train.parquet")
    val_frame = load_parquet(data_dir / "validation.parquet")
    test_frame = load_parquet(data_dir / "test.parquet")

    X_train, X_val, X_test = (transform.transform(f) for f in (train_frame, val_frame, test_frame))
    Y_train, Y_val, Y_test = (f[OUTCOME_COLUMN] for f in (train_frame, val_frame, test_frame))
    T_train, T_val, T_test = (f[TREATMENT_COLUMN] for f in (train_frame, val_frame, test_frame))
    row_id_val, row_id_test = val_frame["row_id"], test_frame["row_id"]
    del train_frame
    gc.collect()

    start = time.perf_counter()
    model = fit_response_model(X_train, Y_train, X_val, Y_val, seed=SEED)
    runtime = time.perf_counter() - start
    print(f"fitted in {runtime:.1f}s (best iteration {model.best_iteration})")

    val_scores = predict(model, X_val)
    test_scores = predict(model, X_test)
    diagnostics = response_diagnostics(val_scores, Y_val)
    val_metrics = evaluate_ranking(val_scores, T_val, Y_val)
    test_metrics = evaluate_ranking(test_scores, T_test, Y_test)

    from sklearn.calibration import calibration_curve
    from sklearn.metrics import precision_recall_curve, roc_curve
    fpr, tpr, _ = roc_curve(Y_val, val_scores)
    precision, recall, _ = precision_recall_curve(Y_val, val_scores)
    frac_pos, mean_pred = calibration_curve(Y_val, val_scores, n_bins=10, strategy="quantile")
    save_csv(pd.DataFrame({"fpr": fpr, "tpr": tpr}), baseline_dir / "roc_curve.csv")
    save_csv(pd.DataFrame({"precision": precision, "recall": recall}), baseline_dir / "pr_curve.csv")
    save_csv(pd.DataFrame({"mean_predicted": mean_pred, "fraction_positive": frac_pos}),
              baseline_dir / "calibration_curve.csv")

    save_pickle(model, baseline_dir / "model.pkl")
    metrics = package_ranking_artifacts(
        baseline_dir, "Response LightGBM", val_metrics, test_metrics,
        val_scores, test_scores, T_val, Y_val, T_test, Y_test,
        row_id_val, row_id_test, runtime,
        extra_metrics={
            **experiment_metadata(OUTCOME_COLUMN, seed=SEED, data_signature=DATA_SIGNATURE),
            "best_iteration": int(model.best_iteration),
            "val_roc_auc": diagnostics.roc_auc, "val_average_precision": diagnostics.average_precision,
            "val_log_loss": diagnostics.log_loss,
        },
    )
    print("validation qini_above_random:", round(val_metrics.qini_above_random, 4))

    del X_train, X_val, X_test, val_frame, test_frame
    gc.collect()
    return metrics


if RUN_STAGE in ("baseline", "all"):
    BASELINE_METRICS = ensure_baseline_artifacts()
else:
    print(f"Baseline stage skipped (RUN_STAGE={RUN_STAGE!r})")

In [ ]:
if RUN_STAGE in ("baseline", "all"):
    baseline_dir = stage_dir("baseline")
    roc = load_csv_artifact(baseline_dir / "roc_curve.csv")
    pr = load_csv_artifact(baseline_dir / "pr_curve.csv")
    cal = load_csv_artifact(baseline_dir / "calibration_curve.csv")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(roc["fpr"], roc["tpr"], color="#2b6cb0")
    axes[0].plot([0, 1], [0, 1], "--", color="#888888")
    axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
    axes[0].set_title(f"ROC (AUC={BASELINE_METRICS['val_roc_auc']:.4f})")

    axes[1].plot(pr["recall"], pr["precision"], color="#2b6cb0")
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].set_title(f"PR (AP={BASELINE_METRICS['val_average_precision']:.4f})")

    axes[2].plot(cal["mean_predicted"], cal["fraction_positive"], "o-", color="#2b6cb0")
    axes[2].plot([0, 1], [0, 1], "--", color="#888888")
    axes[2].set_xlabel("Mean predicted probability"); axes[2].set_ylabel("Observed conversion rate")
    axes[2].set_title("Calibration (validation, 10 quantile bins)")
    fig.suptitle("Response model diagnostics -- validation partition")
    fig.tight_layout()
    plt.show()

## Stage 3 -- Uplift models (T-Learner, X-Learner)

**T-Learner**: two independent LightGBM classifiers, one per arm;
`tau_hat(x) = mu1_hat(x) - mu0_hat(x)`.

**X-Learner**: `fit_x_learner` takes the **raw** train frame deliberately --
it fits a *fold-local* categorical transform inside each of its two
cross-fitting folds. Passing an already-transformed frame would leak each
fold's category vocabulary into the other fold; `fit_x_learner` raises rather
than let that happen silently (a real bug this project found and fixed --
see README's Methodology notes).

**Artifacts written** under `outputs/{OUTCOME_COLUMN}/uplift/tlearner/` and
`outputs/{OUTCOME_COLUMN}/uplift/xlearner/`, each with the same `model.pkl` /
`metrics.json` /
`predictions.parquet` / `qini_curve.csv` / `uplift_curve.csv` /
`decile_table.csv` shape as the baseline stage.

In [ ]:
def ensure_uplift_artifacts():
    uplift_dir = stage_dir("uplift", outcome=OUTCOME_COLUMN)
    tlearner_dir, xlearner_dir = uplift_dir / "tlearner", uplift_dir / "xlearner"
    if (artifact_is_fresh(tlearner_dir / "metrics.json", DATA_SIGNATURE, outcome=OUTCOME_COLUMN)
            and (tlearner_dir / "model.pkl").is_file()
            and artifact_is_fresh(xlearner_dir / "metrics.json", DATA_SIGNATURE, outcome=OUTCOME_COLUMN)
            and (xlearner_dir / "model.pkl").is_file()):
        print(f"Using cached uplift artifacts ({uplift_dir})")
        return load_json(tlearner_dir / "metrics.json"), load_json(xlearner_dir / "metrics.json")

    print("Fitting T-Learner / X-Learner (cache missing or stale) ...")
    data_dir = stage_dir("data")
    transform = ensure_lgbm_transform()
    train_frame = load_parquet(data_dir / "train.parquet")
    val_frame = load_parquet(data_dir / "validation.parquet")
    test_frame = load_parquet(data_dir / "test.parquet")

    X_train, X_val, X_test = (transform.transform(f) for f in (train_frame, val_frame, test_frame))
    Y_train, Y_val, Y_test = (f[OUTCOME_COLUMN] for f in (train_frame, val_frame, test_frame))
    T_train, T_val, T_test = (f[TREATMENT_COLUMN] for f in (train_frame, val_frame, test_frame))
    row_id_val, row_id_test = val_frame["row_id"], test_frame["row_id"]

    tlearner_dir.mkdir(parents=True, exist_ok=True)
    start = time.perf_counter()
    t_learner = fit_t_learner(X_train, T_train, Y_train, X_val, T_val, Y_val, seed=SEED)
    t_runtime = time.perf_counter() - start
    print(f"T-Learner fitted in {t_runtime:.1f}s")
    t_val_scores, t_test_scores = t_learner.predict_tau(X_val), t_learner.predict_tau(X_test)
    t_val_metrics = evaluate_ranking(t_val_scores, T_val, Y_val)
    t_test_metrics = evaluate_ranking(t_test_scores, T_test, Y_test)
    save_pickle(t_learner, tlearner_dir / "model.pkl")
    t_metrics = package_ranking_artifacts(
        tlearner_dir, "T-Learner", t_val_metrics, t_test_metrics,
        t_val_scores, t_test_scores, T_val, Y_val, T_test, Y_test,
        row_id_val, row_id_test, t_runtime,
        extra_metrics=experiment_metadata(OUTCOME_COLUMN, seed=SEED, data_signature=DATA_SIGNATURE),
    )
    print("validation qini_above_random:", round(t_val_metrics.qini_above_random, 4))

    del X_train, X_val, X_test
    gc.collect()

    xlearner_dir.mkdir(parents=True, exist_ok=True)
    start = time.perf_counter()
    x_learner = fit_x_learner(train_frame, T_train, Y_train, seed=SEED)
    x_runtime = time.perf_counter() - start
    print(f"X-Learner fitted in {x_runtime:.1f}s")
    X_val_x, X_test_x = transform.transform(val_frame), transform.transform(test_frame)
    x_val_scores, x_test_scores = x_learner.predict_tau(X_val_x), x_learner.predict_tau(X_test_x)
    x_val_metrics = evaluate_ranking(x_val_scores, T_val, Y_val)
    x_test_metrics = evaluate_ranking(x_test_scores, T_test, Y_test)
    save_pickle(x_learner, xlearner_dir / "model.pkl")
    x_metrics = package_ranking_artifacts(
        xlearner_dir, "X-Learner", x_val_metrics, x_test_metrics,
        x_val_scores, x_test_scores, T_val, Y_val, T_test, Y_test,
        row_id_val, row_id_test, x_runtime,
        extra_metrics=experiment_metadata(OUTCOME_COLUMN, seed=SEED, data_signature=DATA_SIGNATURE),
    )
    print("validation qini_above_random:", round(x_val_metrics.qini_above_random, 4))

    del train_frame, val_frame, test_frame, X_val_x, X_test_x
    gc.collect()
    return t_metrics, x_metrics


if RUN_STAGE in ("uplift", "all"):
    TLEARNER_METRICS, XLEARNER_METRICS = ensure_uplift_artifacts()
else:
    print(f"Uplift stage skipped (RUN_STAGE={RUN_STAGE!r})")

## Stage 4 -- Causal Forest

`econml.grf.CausalForest` has no native categorical support and consumes a
dense numeric matrix, so categorical features go through
`CausalForestCategoricalEncoder`: **frequency-capped top-K one-hot**, with an
`OTHER` bucket for the tail and for categories unseen at train time. Raw
integer tokens would fabricate a false ordering the forest's splits would
exploit -- `fit_causal_forest` rejects that outright.

**This is the slow, memory-heavy stage** (`n_jobs=1`, required for
reproducible predictions). The encoded train matrix is built, fit, and freed
before the validation/test matrices are ever built -- at most one CF-encoded
partition exists in memory at a time. `K = configs/config.yaml:
causal_forest.categorical_top_k` (shipped default `8`) is a resource-ladder
choice, not a free parameter -- see README's Methodology notes for why `K=32`
risks ~21GB for the encoded train matrix alone at full CRITEO scale.

**Artifacts written** under `outputs/{OUTCOME_COLUMN}/causal_forest/`: the usual
`model.pkl` / `metrics.json` / prediction and curve files, plus
`resource_evidence.json` recording the actual `K`, encoded column count, and
an estimated encoded-matrix size -- the resource-gate evidence for whichever
`K` this run actually used.

In [ ]:
def ensure_cf_encoder():
    prep_dir = stage_dir("preprocessing")
    meta_path = prep_dir / "cf_encoder_metadata.json"
    encoder_path = prep_dir / "cf_encoder.joblib"
    k = CONFIG["causal_forest"]["categorical_top_k"]
    if artifact_is_fresh(meta_path, DATA_SIGNATURE) and encoder_path.is_file():
        if load_json(meta_path).get("k") == k:
            return load_pickle(encoder_path)

    train_frame = load_parquet(stage_dir("data") / "train.parquet")
    encoder = CausalForestCategoricalEncoder(k=k).fit(train_frame)
    save_pickle(encoder, encoder_path)
    save_json({"data_signature": DATA_SIGNATURE, "k": k}, meta_path)
    del train_frame
    gc.collect()
    return encoder

In [ ]:
def ensure_causal_forest_artifacts():
    cf_dir = stage_dir("causal_forest", outcome=OUTCOME_COLUMN)
    meta_path = cf_dir / "metrics.json"
    if (artifact_is_fresh(meta_path, DATA_SIGNATURE, outcome=OUTCOME_COLUMN)
            and (cf_dir / "model.pkl").is_file()):
        print(f"Using cached Causal Forest artifacts ({cf_dir})")
        return load_json(meta_path)

    print("Fitting Causal Forest (cache missing or stale; slow stage, n_jobs=1) ...")
    data_dir = stage_dir("data")
    encoder = ensure_cf_encoder()

    train_frame = load_parquet(data_dir / "train.parquet")
    T_train, Y_train = train_frame[TREATMENT_COLUMN], train_frame[OUTCOME_COLUMN]
    X_train_cf = encoder.transform(train_frame)
    encoded_feature_count = int(X_train_cf.shape[1])
    n_train_rows = int(len(X_train_cf))
    del train_frame
    gc.collect()

    start = time.perf_counter()
    model = fit_causal_forest(X_train_cf, T_train, Y_train, seed=SEED)
    runtime = time.perf_counter() - start
    print(f"fitted in {runtime:.1f}s")
    del X_train_cf
    gc.collect()

    val_frame = load_parquet(data_dir / "validation.parquet")
    T_val, Y_val, row_id_val = val_frame[TREATMENT_COLUMN], val_frame[OUTCOME_COLUMN], val_frame["row_id"]
    X_val_cf = encoder.transform(val_frame)
    val_scores = predict_causal_forest_tau(model, X_val_cf)
    val_metrics = evaluate_ranking(val_scores, T_val, Y_val)
    del val_frame, X_val_cf
    gc.collect()

    test_frame = load_parquet(data_dir / "test.parquet")
    T_test, Y_test, row_id_test = test_frame[TREATMENT_COLUMN], test_frame[OUTCOME_COLUMN], test_frame["row_id"]
    X_test_cf = encoder.transform(test_frame)
    test_scores = predict_causal_forest_tau(model, X_test_cf)
    test_metrics = evaluate_ranking(test_scores, T_test, Y_test)
    del test_frame, X_test_cf
    gc.collect()

    save_pickle(model, cf_dir / "model.pkl")
    cf_config = CONFIG["causal_forest"]
    save_json({
        "categorical_top_k": cf_config["categorical_top_k"],
        "encoded_feature_count": encoded_feature_count,
        "n_train_rows": n_train_rows,
        "estimated_encoded_matrix_bytes": n_train_rows * encoded_feature_count * 8,
        "n_estimators": cf_config["n_estimators"], "honest": cf_config["honest"],
        "min_samples_leaf": cf_config["min_samples_leaf"], "max_samples": cf_config["max_samples"],
        "subforest_size": cf_config["subforest_size"], "n_jobs": cf_config["n_jobs"],
        "runtime_seconds": runtime,
    }, cf_dir / "resource_evidence.json")

    metrics = package_ranking_artifacts(
        cf_dir, "Causal Forest", val_metrics, test_metrics,
        val_scores, test_scores, T_val, Y_val, T_test, Y_test,
        row_id_val, row_id_test, runtime,
        extra_metrics=experiment_metadata(OUTCOME_COLUMN, seed=SEED, data_signature=DATA_SIGNATURE),
    )
    print("validation qini_above_random:", round(val_metrics.qini_above_random, 4))
    return metrics


if RUN_STAGE in ("causal_forest", "all"):
    if RUN_CAUSAL_FOREST:
        CAUSAL_FOREST_METRICS = ensure_causal_forest_artifacts()
    else:
        CAUSAL_FOREST_METRICS = None
        print("Causal Forest skipped (RUN_CAUSAL_FOREST = False)")
else:
    print(f"Causal Forest stage skipped (RUN_STAGE={RUN_STAGE!r})")

In [ ]:
if RUN_STAGE in ("causal_forest", "all") and RUN_CAUSAL_FOREST:
    resource_evidence = load_json(stage_dir("causal_forest") / "resource_evidence.json")
    gb = resource_evidence["estimated_encoded_matrix_bytes"] / 1024**3
    print(f"K (categorical_top_k)   : {resource_evidence['categorical_top_k']}")
    print(f"Encoded feature count   : {resource_evidence['encoded_feature_count']}")
    print(f"Train rows encoded      : {resource_evidence['n_train_rows']:,}")
    print(f"Estimated encoded matrix: {gb:.3f} GB (float64, before CausalForest's own fit-time overhead)")
    print(f"n_estimators={resource_evidence['n_estimators']}  honest={resource_evidence['honest']}  "
          f"n_jobs={resource_evidence['n_jobs']}  runtime={resource_evidence['runtime_seconds']:.1f}s")

## Stage 5 -- Final report

**Fits nothing.** Every cell below reads only the artifacts each earlier
stage saved under `outputs/` -- this is the stage to run (`RUN_STAGE =
"report"`) after a kernel restart to see the full comparison without
retraining anything. Whichever of the four models were actually run appear
below; a model whose stage was skipped or never run is shown as unavailable,
never fabricated.

Everything is scored on the **test** partition -- never fit on, early-stopped
against, or used for selection. A random ranking is included as the honest
floor: an uplift model that cannot beat it has not earned its complexity.

In [ ]:
if RUN_STAGE in ("report", "all"):
    MODEL_STAGE_DIRS = {
        "Response LightGBM": stage_dir("baseline", outcome=OUTCOME_COLUMN),
        "T-Learner": stage_dir("uplift", outcome=OUTCOME_COLUMN) / "tlearner",
        "X-Learner": stage_dir("uplift", outcome=OUTCOME_COLUMN) / "xlearner",
        "Causal Forest": stage_dir("causal_forest", outcome=OUTCOME_COLUMN),
    }
    available = {label: d for label, d in MODEL_STAGE_DIRS.items() if (d / "metrics.json").is_file()}
    print("Available model artifacts:", list(available) or "NONE")
    if not available:
        raise RuntimeError(
            "No model artifacts found under outputs/. Run RUN_STAGE in "
            "{'baseline', 'uplift', 'causal_forest', 'all'} at least once first."
        )

    figures_dir = stage_dir("report", outcome=OUTCOME_COLUMN) / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)

    reference_label = next(iter(available))
    reference_predictions = load_parquet(available[reference_label] / "predictions.parquet")
    test_ref = reference_predictions[reference_predictions["partition"] == "test"].sort_values("row_id")
    T_test_ref, Y_test_ref = test_ref["treatment"].to_numpy(), test_ref["outcome"].to_numpy()
    rng = np.random.default_rng(SEED)
    random_metrics = evaluate_ranking(rng.uniform(size=len(T_test_ref)), T_test_ref, Y_test_ref)

    OBJECTIVE_LABELS = {
        "Response LightGBM": "P(Y|X) \u2014 naive targeting baseline",
        "T-Learner": "tau(X) \u2014 CATE estimation",
        "X-Learner": "tau(X) \u2014 CATE estimation",
        "Causal Forest": "tau(X) \u2014 CATE estimation",
        "Random (reference)": "Uninformative random ranking",
    }

    rows = []
    for label, d in available.items():
        m = load_json(d / "metrics.json")
        row = {"model": label, "objective": OBJECTIVE_LABELS[label],
               "auuc_above_random": m["test_auuc_above_random"],
               "qini_above_random": m["test_qini_above_random"],
               "auuc_area": m["test_auuc_area"], "qini_area": m["test_qini_area"]}
        row.update({f"uplift@{k}": v for k, v in m["test_uplift_at_k"].items()})
        rows.append(row)
    rows.append({
        "model": "Random (reference)", "objective": OBJECTIVE_LABELS["Random (reference)"],
        "auuc_above_random": random_metrics.auuc_above_random,
        "qini_above_random": random_metrics.qini_above_random, "auuc_area": random_metrics.auuc_area,
        "qini_area": random_metrics.qini_area,
        **{f"uplift@{k}": v for k, v in random_metrics.uplift_at_k.items()},
    })
    comparison = pd.DataFrame(rows).set_index("model").sort_values("qini_above_random", ascending=False)
    comparison = comparison[["objective"] + [c for c in comparison.columns if c != "objective"]]
    save_csv(comparison.reset_index(), stage_dir("report", outcome=OUTCOME_COLUMN) / "model_comparison.csv")
    display(comparison.round(5))

In [ ]:
if RUN_STAGE in ("report", "all"):
    causal_models = [m for m in ("T-Learner", "X-Learner", "Causal Forest") if m in available]
    ranked = comparison.drop(index="Random (reference)", errors="ignore")
    best = ranked.index[0]

    print("BEST MODEL (by point estimate):", best)
    print(f"  Qini above random : {ranked.loc[best, 'qini_above_random']:.5f}")
    print(f"  AUUC above random : {ranked.loc[best, 'auuc_above_random']:.5f}")
    print(f"  uplift@10pct      : {ranked.loc[best, 'uplift@10pct']}")
    print()
    beat_random = [m for m in ranked.index if ranked.loc[m, "qini_above_random"] > random_metrics.qini_above_random]
    print("Models beating the random reference:", beat_random or "NONE")

    # A higher point estimate alone does not establish superiority -- check
    # whether the gap to the runner-up survives resampling noise before
    # calling `best` the winner.
    if len(ranked.index) > 1:
        runner_up = ranked.index[1]
        def _test_scores(label):
            p = load_parquet(available[label] / "predictions.parquet")
            p = p[p["partition"] == "test"].sort_values("row_id")
            return p["score"].to_numpy(dtype=np.float64)
        score_best, score_runner_up = _test_scores(best), _test_scores(runner_up)

        rng_boot = np.random.default_rng(SEED)
        n = len(T_test_ref)
        N_BOOT_TOP2 = 500
        gap = np.empty(N_BOOT_TOP2)
        for b in range(N_BOOT_TOP2):
            idx = rng_boot.integers(0, n, size=n)
            m_best = evaluate_ranking(score_best[idx], T_test_ref[idx], Y_test_ref[idx])
            m_runner_up = evaluate_ranking(score_runner_up[idx], T_test_ref[idx], Y_test_ref[idx])
            gap[b] = m_best.qini_above_random - m_runner_up.qini_above_random
        lo, hi = np.percentile(gap, [2.5, 97.5])
        significant = lo > 0 or hi < 0
        print()
        print(f"Paired bootstrap ({N_BOOT_TOP2} resamples): {best} vs {runner_up}, "
              f"Qini-above-random gap 95% CI = [{lo:.4f}, {hi:.4f}]")
        if significant:
            print(f"  -> gap is statistically distinguishable from resampling noise at this sample size.")
        else:
            print(f"  -> gap is NOT statistically distinguishable from resampling noise at this sample "
                  f"size -- {best} should not be reported as definitively superior to {runner_up} on this basis alone.")

### Uplift visualization

The curves below show the full ranking, not just the single-number summary
in the table above -- a model can lead on Qini-above-random while still
crossing below another model (or the random reference) over part of the
coverage range, which the table alone would hide.

In [ ]:
if RUN_STAGE in ("report", "all"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for label, d in available.items():
        qc, uc = load_csv_artifact(d / "qini_curve.csv"), load_csv_artifact(d / "uplift_curve.csv")
        plot_label = "Response LightGBM (naive targeting reference)" if label == "Response LightGBM" else label
        ax1.plot(qc["coverage"], qc["qini_gain"], label=plot_label)
        ax2.plot(uc["coverage"], uc["uplift_gain"], label=plot_label)
    ax1.plot(random_metrics.qini_curve["coverage"], random_metrics.qini_curve["qini_gain"],
              "--", color="#888888", label="Random (reference)")
    ax2.plot(random_metrics.uplift_curve["coverage"], random_metrics.uplift_curve["uplift_gain"],
              "--", color="#888888", label="Random (reference)")
    ax1.set_xlabel("Coverage"); ax1.set_ylabel("Qini gain"); ax1.set_title("Qini curves (test)"); ax1.legend()
    ax2.set_xlabel("Coverage"); ax2.set_ylabel("Uplift gain"); ax2.set_title("Uplift curves / AUUC (test)"); ax2.legend()
    fig.tight_layout()
    fig.savefig(figures_dir / "qini_uplift_curves.png", dpi=120)
    plt.show()

In [ ]:
if RUN_STAGE in ("report", "all"):
    fig, ax = plt.subplots(figsize=(9, 4))
    comparison[["qini_above_random", "auuc_above_random"]].plot.barh(
        ax=ax, title="Test-set ranking performance above the random reference"
    )
    fig.tight_layout()
    fig.savefig(figures_dir / "comparison_barh.png", dpi=120)
    plt.show()

### Treatment effect distribution

A model whose predicted effect is nearly constant is not finding heterogeneity, whatever its Qini says.

In [ ]:
if RUN_STAGE in ("report", "all"):
    if causal_models:
        cate = pd.DataFrame({
            label: load_parquet(available[label] / "predictions.parquet")
                .pipe(lambda f: f[f["partition"] == "test"].sort_values("row_id"))["score"].to_numpy()
            for label in causal_models
        })
        display(cate.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T.round(6))

        fig, axes = plt.subplots(1, len(causal_models), figsize=(5 * len(causal_models), 3.5), squeeze=False)
        for ax, label in zip(axes[0], causal_models):
            ax.hist(cate[label], bins=60, color="#2b6cb0")
            ax.axvline(0.0, color="k", linestyle="--", linewidth=1)
            ax.set_title(f"{label}\npredicted CATE")
        fig.tight_layout()
        fig.savefig(figures_dir / "cate_distribution.png", dpi=120)
        plt.show()
    else:
        print("No causal model (T-Learner / X-Learner / Causal Forest) artifacts available yet.")

### Uplift decile analysis

If a ranking is real, observed uplift should decline from decile 1 (highest predicted) downward.

In [ ]:
if RUN_STAGE in ("report", "all") and causal_models:
    deciles = pd.DataFrame({
        label: load_csv_artifact(available[label] / "decile_table.csv").set_index("decile")["observed_uplift"]
        for label in causal_models
    })
    display(deciles.round(5))

    ax = deciles.plot(marker="o", figsize=(9, 4),
                       title="Observed test uplift by predicted-CATE decile (1 = highest predicted)")
    ax.axhline(0.0, color="k", linestyle="--", linewidth=1)
    ax.set_xlabel("Decile of predicted CATE"); ax.set_ylabel("Observed uplift")
    ax.figure.savefig(figures_dir / "uplift_deciles.png", dpi=120)
    plt.show()

    if len(causal_models) > 1:
        print("Spearman rank correlation between causal models' predicted CATE:")
        print(cate.corr(method="spearman").round(3))

### Response model diagnostics (ROC / PR / calibration)

Reproduced here from the baseline stage's saved artifacts so the report stage stays self-contained after a kernel restart.

In [ ]:
if RUN_STAGE in ("report", "all"):
    if "Response LightGBM" in available:
        baseline_dir = available["Response LightGBM"]
        roc = load_csv_artifact(baseline_dir / "roc_curve.csv")
        pr = load_csv_artifact(baseline_dir / "pr_curve.csv")
        cal = load_csv_artifact(baseline_dir / "calibration_curve.csv")
        baseline_metrics = load_json(baseline_dir / "metrics.json")

        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        axes[0].plot(roc["fpr"], roc["tpr"], color="#2b6cb0")
        axes[0].plot([0, 1], [0, 1], "--", color="#888888")
        axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
        axes[0].set_title(f"ROC (AUC={baseline_metrics['val_roc_auc']:.4f})")
        axes[1].plot(pr["recall"], pr["precision"], color="#2b6cb0")
        axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
        axes[1].set_title(f"PR (AP={baseline_metrics['val_average_precision']:.4f})")
        axes[2].plot(cal["mean_predicted"], cal["fraction_positive"], "o-", color="#2b6cb0")
        axes[2].plot([0, 1], [0, 1], "--", color="#888888")
        axes[2].set_xlabel("Mean predicted probability"); axes[2].set_ylabel("Observed conversion rate")
        axes[2].set_title("Calibration (validation)")
        fig.suptitle("Response model diagnostics -- validation partition")
        fig.tight_layout()
        fig.savefig(figures_dir / "baseline_diagnostics.png", dpi=120)
        plt.show()
    else:
        print("Response model artifacts not available.")

## Causal Forest interpretation

Causal Forest gets its own pass because its `K=8` categorical encoding is a
memory/runtime tradeoff, not a performance choice (see Stage 4) -- worth
checking its predicted-CATE spread and decile behavior on its own before
comparing it to the other estimators above.

In [ ]:
if RUN_STAGE in ("report", "all"):
    if "Causal Forest" in available:
        cf_metrics = load_json(available["Causal Forest"] / "metrics.json")
        cf_deciles = load_csv_artifact(available["Causal Forest"] / "decile_table.csv").sort_values("decile")
        cf_resource = load_json(stage_dir("causal_forest", outcome=OUTCOME_COLUMN) / "resource_evidence.json")
        cf_cate = cate["Causal Forest"]
        top_decile, bottom_decile = cf_deciles.iloc[0], cf_deciles.iloc[-1]

        print(f"Test qini_above_random  : {cf_metrics['test_qini_above_random']:.5f}")
        print(f"Mean / median predicted CATE : {cf_cate.mean():.6f} / {cf_cate.median():.6f}")
        print(f"Share of rows with predicted CATE > 0 : {(cf_cate > 0).mean():.3%}")
        print(f"Top decile (highest predicted) observed uplift    : {top_decile['observed_uplift']:.5f}  (n={int(top_decile['n'])})")
        print(f"Bottom decile (lowest predicted) observed uplift  : {bottom_decile['observed_uplift']:.5f}  (n={int(bottom_decile['n'])})")
        print(f"Categorical representation: K={cf_resource['categorical_top_k']} -> "
              f"{cf_resource['encoded_feature_count']} encoded columns -- coarser than the other estimators' "
              "full-cardinality LightGBM categorical splits (see README's Methodology notes).")
    else:
        print(
            "Not available -- Causal Forest resource gate/model execution is pending. "
            "Set RUN_CAUSAL_FOREST = True, RUN_STAGE in {'causal_forest', 'all'}, and re-run."
        )

## Sensitivity Analysis: Alternative Outcome Definition (Visit)

**Fits nothing here either.** Like the rest of Stage 5, this section only
reads whatever `outputs/conversion/` and `outputs/visit/` artifacts already
exist on disk -- produced by running this notebook once with
`OUTCOME = "conversion"` and once with `OUTCOME = "visit"` (`RUN_STAGE =
"all"` each time). If one of the two hasn't been run yet, that half is
reported as unavailable below, never fabricated.

**Observation.** `conversion` is the primary, business-oriented outcome this
project optimizes for -- but it is a rare event (see Stage 1's outcome
comparison). `visit` is a denser behavioral outcome on the same causal path
(every conversion is preceded by a visit), used here purely as a
**sensitivity/robustness check**: does the model ranking established under
`conversion` hold up under a less sparse outcome definition, or could it be
an artifact of how rare conversion is? `visit` never replaces `conversion`
as the primary result -- it runs on the exact same physical
train/validation/test row partition (see Stage 1); only the `Y` column
differs.

**Quantitative evidence.**

In [ ]:
if RUN_STAGE in ("report", "all"):
    sensitivity_dataset_summary = load_json(stage_dir("data") / "dataset_summary.json")

    def _prevalence_row(label, overall_key, by_treatment_key):
        rates = sensitivity_dataset_summary[by_treatment_key]
        return {
            "outcome": label,
            "prevalence": sensitivity_dataset_summary[overall_key],
            "treatment_rate": rates.get("1", float("nan")),
            "control_rate": rates.get("0", float("nan")),
        }

    outcome_prevalence_table = pd.DataFrame([
        _prevalence_row("Conversion", "conversion_rate", "conversion_rate_by_treatment"),
        _prevalence_row("Visit", "visit_rate", "visit_rate_by_treatment"),
    ]).set_index("outcome")
    display(outcome_prevalence_table.round(6))

In [ ]:
if RUN_STAGE in ("report", "all"):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    outcome_prevalence_table["prevalence"].plot.bar(ax=ax, color=["#2b6cb0", "#c05621"])
    ax.set_ylabel("Overall rate")
    ax.set_title("Outcome prevalence: conversion vs. visit")
    ax.tick_params(axis="x", rotation=0)
    for i, v in enumerate(outcome_prevalence_table["prevalence"]):
        ax.text(i, v, f"{v:.4f}", ha="center", va="bottom")
    fig.tight_layout()
    fig.savefig(figures_dir / "sensitivity_outcome_prevalence.png", dpi=120)
    plt.show()

In [ ]:
if RUN_STAGE in ("report", "all"):
    SENSITIVITY_OBJECTIVE_LABELS = {
        "Response LightGBM": "P(Y|X) \u2014 naive targeting baseline",
        "T-Learner": "tau(X) \u2014 CATE estimation",
        "X-Learner": "tau(X) \u2014 CATE estimation",
        "Causal Forest": "tau(X) \u2014 CATE estimation",
    }

    def _load_outcome_metrics(outcome_name):
        stage_map = {
            "Response LightGBM": stage_dir("baseline", outcome=outcome_name),
            "T-Learner": stage_dir("uplift", outcome=outcome_name) / "tlearner",
            "X-Learner": stage_dir("uplift", outcome=outcome_name) / "xlearner",
            "Causal Forest": stage_dir("causal_forest", outcome=outcome_name),
        }
        rows = []
        for model_label, d in stage_map.items():
            meta_path = d / "metrics.json"
            if not meta_path.is_file():
                continue
            m = load_json(meta_path)
            rows.append({
                "outcome": outcome_name, "model": model_label,
                "objective": SENSITIVITY_OBJECTIVE_LABELS[model_label],
                "qini_above_random": m["test_qini_above_random"],
                "auuc_above_random": m["test_auuc_above_random"],
                "uplift_at_10pct": m["test_uplift_at_k"]["10pct"],
            })
        return rows

    sensitivity_rows = _load_outcome_metrics("conversion") + _load_outcome_metrics("visit")
    by_outcome_comparison = pd.DataFrame(
        sensitivity_rows,
        columns=["outcome", "model", "objective", "qini_above_random", "auuc_above_random", "uplift_at_10pct"],
    )
    available_sensitivity_outcomes = (
        sorted(by_outcome_comparison["outcome"].unique()) if not by_outcome_comparison.empty else []
    )
    missing_sensitivity_outcomes = [o for o in ("conversion", "visit") if o not in available_sensitivity_outcomes]
    if missing_sensitivity_outcomes:
        print(f"Not yet available: {missing_sensitivity_outcomes} -- run RUN_STAGE='all' with OUTCOME set to "
              f"each missing value first. Showing only what has been run.")
    if by_outcome_comparison.empty:
        print("No model artifacts found for either outcome yet.")
    else:
        display(by_outcome_comparison.set_index(["outcome", "model"]).round(5))

In [ ]:
if RUN_STAGE in ("report", "all"):
    if set(available_sensitivity_outcomes) >= {"conversion", "visit"}:
        pivot_qini = by_outcome_comparison.pivot(index="model", columns="outcome", values="qini_above_random")
        pivot_auuc = by_outcome_comparison.pivot(index="model", columns="outcome", values="auuc_above_random")
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
        pivot_qini.plot.bar(ax=ax1, color=["#2b6cb0", "#c05621"])
        ax1.set_title("Qini above random by model and outcome"); ax1.set_ylabel("Qini above random")
        ax1.tick_params(axis="x", rotation=30)
        pivot_auuc.plot.bar(ax=ax2, color=["#2b6cb0", "#c05621"])
        ax2.set_title("AUUC above random by model and outcome"); ax2.set_ylabel("AUUC above random")
        ax2.tick_params(axis="x", rotation=30)
        fig.text(0.5, -0.02, "Point estimates only -- see paired bootstrap below for statistical uncertainty.",
                 ha="center", fontsize=8, style="italic")
        fig.tight_layout()
        fig.savefig(figures_dir / "sensitivity_model_ranking_by_outcome.png", dpi=120, bbox_inches="tight")
        plt.show()
    else:
        print("Model-ranking-by-outcome comparison needs both outcomes' artifacts -- skipping plot.")

In [ ]:
if RUN_STAGE in ("report", "all"):
    if set(available_sensitivity_outcomes) >= {"conversion", "visit"}:
        pivot_uplift10 = by_outcome_comparison.pivot(index="model", columns="outcome", values="uplift_at_10pct")
        fig, ax = plt.subplots(figsize=(7, 4.5))
        pivot_uplift10.plot.bar(ax=ax, color=["#2b6cb0", "#c05621"])
        ax.set_title("uplift@10% by model and outcome"); ax.set_ylabel("uplift@10%")
        ax.tick_params(axis="x", rotation=30)
        fig.text(0.5, -0.02, "Point estimates only -- see paired bootstrap below for statistical uncertainty.",
                 ha="center", fontsize=8, style="italic")
        fig.tight_layout()
        fig.savefig(figures_dir / "sensitivity_uplift_at_10pct_by_outcome.png", dpi=120, bbox_inches="tight")
        plt.show()
    else:
        print("uplift@10% comparison needs both outcomes' artifacts -- skipping plot.")

In [ ]:
if RUN_STAGE in ("report", "all"):
    outcomes_with_curves = [o for o in ("conversion", "visit") if o in available_sensitivity_outcomes]
    if outcomes_with_curves:
        fig, axes = plt.subplots(1, len(outcomes_with_curves), figsize=(6.5 * len(outcomes_with_curves), 5), squeeze=False)
        for ax, outcome_name in zip(axes[0], outcomes_with_curves):
            stage_map = {
                "Response LightGBM": stage_dir("baseline", outcome=outcome_name),
                "T-Learner": stage_dir("uplift", outcome=outcome_name) / "tlearner",
                "X-Learner": stage_dir("uplift", outcome=outcome_name) / "xlearner",
                "Causal Forest": stage_dir("causal_forest", outcome=outcome_name),
            }
            for model_label, d in stage_map.items():
                qc_path = d / "qini_curve.csv"
                if qc_path.is_file():
                    qc = load_csv_artifact(qc_path)
                    plot_label = (
                        "Response LightGBM (naive targeting reference)"
                        if model_label == "Response LightGBM" else model_label
                    )
                    ax.plot(qc["coverage"], qc["qini_gain"], label=plot_label)
            ax.set_xlabel("Coverage"); ax.set_ylabel("Qini gain")
            ax.set_title(f"Qini curve -- {outcome_name}")
            ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(figures_dir / "sensitivity_qini_curves_by_outcome.png", dpi=120)
        plt.show()
    else:
        print("No Qini curve artifacts available for either outcome yet.")

In [ ]:
if RUN_STAGE in ("report", "all"):
    N_BOOT_SENSITIVITY = 500
    for outcome_name in ("conversion", "visit"):
        resp_dir = stage_dir("baseline", outcome=outcome_name)
        cf_dir_o = stage_dir("causal_forest", outcome=outcome_name)
        resp_path, cf_path = resp_dir / "predictions.parquet", cf_dir_o / "predictions.parquet"
        if not (resp_path.is_file() and cf_path.is_file()):
            print(f"[{outcome_name}] Response and/or Causal Forest predictions not available yet -- skipping bootstrap.")
            continue

        resp_pred = load_parquet(resp_path)
        cf_pred = load_parquet(cf_path)
        resp_pred = resp_pred[resp_pred["partition"] == "test"].sort_values("row_id").reset_index(drop=True)
        cf_pred = cf_pred[cf_pred["partition"] == "test"].sort_values("row_id").reset_index(drop=True)
        assert (resp_pred["row_id"].to_numpy() == cf_pred["row_id"].to_numpy()).all(), (
            f"[{outcome_name}] Response/Causal Forest test rows are misaligned"
        )

        T_o = resp_pred["treatment"].to_numpy(dtype=np.float64)
        Y_o = resp_pred["outcome"].to_numpy(dtype=np.float64)
        S_resp_o = resp_pred["score"].to_numpy(dtype=np.float64)
        S_cf_o = cf_pred["score"].to_numpy(dtype=np.float64)
        n_o = len(T_o)

        rng_sensitivity = np.random.default_rng(SEED)
        diffs = {"qini": [], "auuc": [], "u10": []}
        for _ in range(N_BOOT_SENSITIVITY):
            idx = rng_sensitivity.integers(0, n_o, size=n_o)
            Tb, Yb = T_o[idx], Y_o[idx]
            mr = evaluate_ranking(S_resp_o[idx], Tb, Yb)
            mc = evaluate_ranking(S_cf_o[idx], Tb, Yb)
            diffs["qini"].append(mr.qini_above_random - mc.qini_above_random)
            diffs["auuc"].append(mr.auuc_above_random - mc.auuc_above_random)
            diffs["u10"].append(mr.uplift_at_k["10pct"] - mc.uplift_at_k["10pct"])

        print(f"=== {outcome_name} -- Response vs. Causal Forest, {N_BOOT_SENSITIVITY} paired resamples (n_test={n_o}) ===")
        for metric_key, metric_label in [("qini", "Qini"), ("auuc", "AUUC"), ("u10", "uplift@10%")]:
            arr = np.asarray(diffs[metric_key])
            lo, hi = np.percentile(arr, [2.5, 97.5])
            includes_zero = lo <= 0 <= hi
            verdict = "CI includes zero -- not statistically significant" if includes_zero else "CI excludes zero"
            print(f"  {metric_label:10s}: Resp-CF 95% CI=[{lo:.5f}, {hi:.5f}]  ({verdict})")

**Modeling implication.** `conversion`'s rarity (~0.3% base rate in this
dataset) means every stage that estimates a treatment effect by differencing
two outcome models (T-Learner) or regressing an imputed pseudo-outcome
(X-Learner) works with very few positive labels per arm -- ranking
differences between estimators are themselves close to the noise floor.
`visit`'s much higher prevalence (~4.7%) gives the same estimators,
completely unchanged, a denser signal, which is what lets a paired bootstrap
actually start to distinguish them.

**Conclusion.**

- **Conversion (primary, sparse outcome).** Response LightGBM's point-estimate
  Qini, AUUC, and uplift@10% are higher than Causal Forest's, but the paired
  bootstrap above puts the Response-minus-Causal-Forest gap's 95% CI across
  zero on all three metrics. **Response and Causal Forest are statistically
  comparable uplift-ranking performers under conversion** -- the point-estimate
  gap is not distinguishable from resampling noise at this sample size, and
  Response should not be reported as the proven winner.
- **Visit (secondary, denser outcome).** Causal Forest's uplift@10% point
  estimate is higher than Response's here, while Response keeps a lead on
  Qini/AUUC. **Causal Forest shows stronger uplift@10% performance under the
  denser visit outcome** -- but per the bootstrap above, that specific gap's
  95% CI still includes zero, so it is reported as a directional signal
  worth more data, not a proven effect. Do not read either outcome's result
  as a settled statistical win for one model.
- Neither result changes the primary conclusion: `conversion` remains the
  business-relevant experiment, and `visit` is reported here strictly as a
  robustness check on whether that conclusion depends on outcome sparsity --
  it does not appear to.

## Final Summary

State the result in the form the evidence actually supports:

> Among the evaluated implementations, **the model with the highest
> point-estimate ranking score** achieved the strongest *point-estimate*
> test-set uplift ranking on CRITEO-UPLIFTv2.1 under this protocol -- check
> the paired bootstrap gap printed alongside it before reading that as
> proven superiority.

**Practical interpretation.** The most informative row is usually Response
LightGBM against the causal estimators: it separates *who converts* from
*who converts because of the treatment*, which is the entire premise of
uplift modeling. A causal model that cannot beat the random reference has
not earned its added complexity over simply targeting everyone (or no one).
A higher point estimate that is not statistically distinguishable from the
runner-up, however, is not evidence of superiority either -- see the
bootstrap check above, and the outcome-level detail in the Sensitivity
Analysis section: under `conversion`, Response and Causal Forest are
statistically comparable uplift-ranking performers, not a clear win for
either.

**Synthesis.** Outcome sparsity influences uplift estimation difficulty and
relative model behavior: `conversion`'s rarity is why Response and Causal
Forest end up statistically indistinguishable, and `visit`'s higher
prevalence is what lets Causal Forest's uplift@10% point estimate pull
ahead at all -- though, per the sensitivity analysis, not yet as a
statistically proven effect.

**Limitations:**

- **Offline evaluation only.** Qini/AUUC are computed against the historical
  A/B test's logged outcomes -- there is no online experiment confirming that
  acting on these rankings would reproduce the measured incremental effect.
- **Bootstrap coverage is partial.** The paired bootstrap above covers the
  point-estimate top-two gap, and the Sensitivity Analysis section covers
  Response vs. Causal Forest under each outcome -- not every pairwise
  metric difference. A full bootstrap across every model pair and metric
  would be the natural next extension.
- **Computational constraints shaped the design**, not just the science: the
  Causal Forest's `K=8` categorical cap is a memory/runtime tradeoff (see
  Stage 4), not a performance-tuned choice, and gives it a coarser
  categorical representation than the LightGBM-based estimators.
- **Not a validated causal mechanism.** `f0`-`f11` are anonymized with no
  known business meaning, and predicted CATE is not a true individual
  treatment effect -- both potential outcomes are never observed for any one
  row, which is also why no PEHE against ground truth is reported.
- **Specific to this dataset and these implementations.** One dataset, one
  implementation of each method, one hyperparameter setting -- a valid
  conclusion has the form "estimator A ranked incremental conversion better
  than estimator B *here*," not a universal claim about either method.

**Next steps:** a paired bootstrap confidence interval on the Qini gap
between the top two models; an online holdout experiment on the model's
actual targeting decisions; and, if resources allow, a measured
memory/runtime benchmark at `K=16`/`K=32` for the Causal Forest encoding.

In [ ]:
if RUN_STAGE in ("report", "all"):
    print("=" * 60)
    print("FINAL SUMMARY")
    print("=" * 60)
    best_objective = (
        "P(Y|X) -- naive targeting baseline" if best == "Response LightGBM"
        else "tau(X) -- CATE estimation"
    )
    print(f"Highest point estimate : {best}  ({best_objective})")
    print(f"  Qini above random    : {ranked.loc[best, 'qini_above_random']:.5f}")
    print(f"  AUUC above random    : {ranked.loc[best, 'auuc_above_random']:.5f}")
    print(f"  uplift@10pct         : {ranked.loc[best, 'uplift@10pct']}")
    if len(ranked.index) > 1:
        significance_note = (
            "statistically significant" if significant
            else "NOT statistically significant (95% CI includes zero) -- see paired bootstrap above"
        )
        print(f"  vs. runner-up ({runner_up}), paired bootstrap: {significance_note}")
    print(f"Models beating random  : {beat_random or 'NONE'}")
    print(f"Models evaluated       : {list(available)}")
    missing = [m for m in MODEL_STAGE_DIRS if m not in available]
    print(f"Models not yet run     : {missing or 'none'}")
    print(f"Report artifacts       : {stage_dir('report', outcome=OUTCOME_COLUMN)}")